# Detecção de anomalias em Licitações públicas

**Importando bibliotecas**

In [8]:
from torch.utils.data import Dataset
import sys
import zipfile
import csv
import unicodedata
from pathlib import Path
import numpy as np
from numpy.lib import recfunctions as rfn
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from typing import Tuple
from typing import List, Optional, Tuple
import glob
import os
import pandas as pd

In [9]:
# Verificando a Versão do PyTorch
print(f"Versão do Pytorch: {torch.__version__}")

# Verificando a disponibildade da GPU
print(f"Disponibilidade da GPU: {"sim" if torch.cuda.is_available() else "não"} ")

Versão do Pytorch: 2.12.1+cpu
Disponibilidade da GPU: não 


**Configurando o ambiente**

In [10]:
project_root = Path.cwd().resolve().parent
sys.path.insert(0, str(project_root))

RANDOM_SEED = 42

**Carregamento e leitura dos dados**

In [12]:
# Lista para armazenar cada tipo de tabela de todos os ZIPs
licitacoes_list = []
itens_list = []
participantes_list = []
empenhos_list = []

# Encontra todos os arquivos .zip no diretório atual (ex: 202401_Licitacoes.zip, 202402_Licitacoes.zip, etc.)
zip_files = glob.glob(str(project_root / "data" / "*_Licitacoes.zip"))

for zip_path in sorted(zip_files):
    print(f"Lendo o arquivo: {zip_path}")
    with zipfile.ZipFile(zip_path, "r") as z:
        for filename in z.namelist():
            if filename.endswith(".csv"):
                with z.open(filename) as f:
                    df = pd.read_csv(f, sep=";", encoding="latin-1", low_memory=False)

                    # Verifica do nome mais específico para o mais geral
                    if "EmpenhosRelacionados" in filename:
                        empenhos_list.append(df)
                    elif "ParticipantesLicitação" in filename or "ParticipantesLicitacao" in filename:
                        participantes_list.append(df)
                    elif "ItemLicitação" in filename or "ItemLicitacao" in filename:
                        itens_list.append(df)
                    elif "Licitação.csv" in filename or "Licitacao.csv" in filename:
                        licitacoes_list.append(df)

# Consolida em DataFrames únicos
df_licitacoes = pd.concat(licitacoes_list, ignore_index=True) if licitacoes_list else pd.DataFrame()
df_itens = pd.concat(itens_list, ignore_index=True) if itens_list else pd.DataFrame()
df_participantes = pd.concat(participantes_list, ignore_index=True) if participantes_list else pd.DataFrame()
df_empenhos = pd.concat(empenhos_list, ignore_index=True) if empenhos_list else pd.DataFrame()

print("\n--- Carregamento Concluído! ---")
print(f"Total de Licitações: {len(df_licitacoes)}")
print(f"Total de Itens: {len(df_itens)}")
print(f"Total de Participantes: {len(df_participantes)}")
print(f"Total de Empenhos: {len(df_empenhos)}")

Lendo o arquivo: D:\POS_DPLRNG-PROJETOS\eng-soft-ia-fw-2026\data\202401_Licitacoes.zip
Lendo o arquivo: D:\POS_DPLRNG-PROJETOS\eng-soft-ia-fw-2026\data\202402_Licitacoes.zip
Lendo o arquivo: D:\POS_DPLRNG-PROJETOS\eng-soft-ia-fw-2026\data\202403_Licitacoes.zip
Lendo o arquivo: D:\POS_DPLRNG-PROJETOS\eng-soft-ia-fw-2026\data\202404_Licitacoes.zip

--- Carregamento Concluído! ---
Total de Licitações: 8064
Total de Itens: 175689
Total de Participantes: 508824
Total de Empenhos: 0


In [15]:
df_licitacoes.describe()

,Número Licitação,Código UG,Código Modalidade Compra,Código Órgão Superior,Código Órgão
count,8.064000e+03,8064.000000,8064.000000,8064.000000,8064.000000
mean,3.366241e+08,249730.399802,4901.878596,38432.560888,37589.358631
std,4.345168e+08,207149.521843,4996.041888,12978.584963,13518.050810
min,1.202300e+04,30100.000000,1.000000,3000.000000,3000.000000
25%,3.220230e+05,155009.000000,5.000000,26000.000000,26414.000000
50%,1.192023e+06,160086.000000,7.000000,36000.000000,32314.000000
75%,9.000220e+08,240108.750000,9999.000000,52000.000000,52121.000000
max,9.999320e+08,930182.000000,9999.000000,96111.000000,96111.000000


In [18]:
df_itens.describe()

,Número Licitação,Código UG,Código Modalidade Compra,Código Órgão,Quantidade Item
count,1.756890e+05,175689.000000,175689.000000,175689.000000,1.756890e+05
mean,1.530960e+08,224973.917104,8879.785746,42070.044624,1.283943e+04
std,3.372118e+08,191211.713789,3151.566362,12583.094824,5.894648e+05
min,1.202300e+04,30100.000000,1.000000,3000.000000,1.000000e+00
25%,1.420230e+05,154618.000000,9999.000000,26443.000000,1.500000e+01
50%,4.320230e+05,160088.000000,9999.000000,52111.000000,1.000000e+02
75%,1.212023e+06,160468.000000,9999.000000,52121.000000,6.000000e+02
max,9.999320e+08,930182.000000,9999.000000,96111.000000,1.150000e+08


In [22]:
df_participantes.describe()

,Número Licitação,Código UG,Código Modalidade Compra,Código Órgão
count,5.088240e+05,508824.000000,508824.000000,508824.000000
mean,5.405099e+07,266000.531500,8909.782155,42527.417115
std,2.121215e+08,233459.623111,3114.310729,12797.788379
min,1.202300e+04,30100.000000,1.000000,3000.000000
25%,1.220230e+05,154040.000000,9999.000000,26436.000000
50%,3.320220e+05,160106.000000,9999.000000,52111.000000
75%,7.420230e+05,200005.000000,9999.000000,52121.000000
max,9.999320e+08,930182.000000,9999.000000,96111.000000


In [19]:
df_empenhos.describe()

,Número Licitação,Código UG,Nome UG,Código Modalidade Compra,Modalidade Compra,Número Processo,Código Empenho,Data Emissão Empenho,Observação Empenho,Valor Empenho (R$)
count,0,0,0,0,0,0,0,0,0,0
unique,0,0,0,0,0,0,0,0,0,0
top,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


**Análise exploratória dos dados**

In [24]:
print("Shape de Licitações: ", df_licitacoes.shape)
print("Shape de Itens: ", df_itens.shape)
print("Shape de Participantes: ", df_participantes.shape)
print("Shape de Emprenhos: ", df_empenhos.shape)

Shape de Licitações:  (8064, 17)
Shape de Itens:  (175689, 14)
Shape de Participantes:  (508824, 13)
Shape de Emprenhos:  (0, 10)


**Pré-processamento dos dados**

**Separação dos conjuntos de treinamento e de teste**